# A large scale analysis of conversation data across myriad spoken corpora

In [1]:
import os
import re
import numpy as np
import pandas as pd
from tqdm.notebook import tqdm
from datetime import datetime as dt

import warnings; warnings.filterwarnings('ignore')

In [2]:
DATA_LOC = '../data'

# path to processed and ready data
DATA_PATH = os.path.join(DATA_LOC, 'lme_ready')

# path to save data to on completion
VIS_PATH = os.path.join(DATA_LOC, 'web_vis')

# post processing & ready for data visualization steps
FINAL_DOC = os.path.join(DATA_LOC, 'resids.parquet')
print(bool(os.path.exists(FINAL_DOC)))

True


In [3]:
def grab_all_files(PATH, file_type='.csv'):
    files = [
        [
            os.path.join(root, f) for f in files 
            if f.endswith(file_type) and (not f.startswith('._'))
        ] 
        for root, _, files in os.walk(PATH) 
    ]
    return sum(files, [])

#### Load Data

In [4]:
# df = pd.read_table(FINAL_DOC, sep='\t')
df = pd.read_parquet(FINAL_DOC)
df.shape

(15310889, 18)

In [5]:
# to rename the corpus correctly . . . 
def lang(x):
    return x.split('-')[1]

In [6]:
df['lang'] = [lang(x) for x in tqdm(df['corpus'])]

  0%|          | 0/15310889 [00:00<?, ?it/s]

In [7]:
df['conversation_id'].nunique()

1285

In [8]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,lang,groups,sample_delta,lang_,groups_,g,resid
0,2.765743,12.0,10.0,2.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,1,9,356343,0,-0.069038
1,2.836436,12.0,7.0,6.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,2,9,356343,0,-0.074234
2,2.906804,12.0,8.0,9.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,3,9,356343,0,0.021430
3,3.063219,12.0,5.0,10.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,4,9,356343,0,0.101956
4,2.971834,12.0,9.0,12.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,5,9,356343,0,0.111756


### Other data preparation processes

In [9]:
print(df['groups'].str.contains('null-').sum())
df['groups'] = [it.replace('null-', '') for it in tqdm(df['groups'])]

1245241


  0%|          | 0/15310889 [00:00<?, ?it/s]

In [10]:
df.head()

,Hxy,nx,ny,turn_delta,self,x_speaker,dyad,corpus,conversation_id,x_turn_id,null,lang,groups,sample_delta,lang_,groups_,g,resid
0,2.765743,12.0,10.0,2.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,1,9,356343,0,-0.069038
1,2.836436,12.0,7.0,6.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,2,9,356343,0,-0.074234
2,2.906804,12.0,8.0,9.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,3,9,356343,0,0.021430
3,3.063219,12.0,5.0,10.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,4,9,356343,0,0.101956
4,2.971834,12.0,9.0,12.0,False,croation-cro-croation-cro-2011_56-AUM,croation-cro-croation-cro-2011_56-AUM-croation...,croation-cro,croation-cro-croation-cro-2011_56,croation-cro-croation-cro-2011_56-5,False,cro,croation-cro-croation-cro-2011_56-5,5,9,356343,0,0.111756


## Convert values to numeric tags

In [11]:
convert_to_numeric_id = [
    #'x_speaker', 'y_speaker',
    # 'dyad',
    'lang',
    'groups'
    #'x_speaker_turn'
]

In [12]:
for col in convert_to_numeric_id:
    vals = np.unique(df[col].values)
    conversion = {k: i+1 for i,k in enumerate(np.random.choice(vals, size=(len(vals),), replace=False))}

    if isinstance(col,list):
        for c  in col:
            df[c+'_'] = [conversion[v] for v in tqdm(df[c])]

    else:
        df[col+'_'] = [conversion[v] for v in tqdm(df[col])]


  0%|          | 0/15310889 [00:00<?, ?it/s]

  0%|          | 0/15310889 [00:00<?, ?it/s]

## LME Regression: Predicting CE

In [13]:
import statsmodels.formula.api as smf

In [ ]:
# df = df.loc[~df['null']]

### Resid model

In [14]:
df['g'] = 0

In [15]:
##########################################
## A set of resids to show ∇H / t
##########################################
model = "Hxy ~ nx + ny + null + (1|groups_) + (2|lang_)"
##########################################

In [16]:
start = dt.now()
# md = smf.mixedlm(model, data=df, groups=df['x_speaker_turn'])
md = smf.mixedlm(model, data=df, groups=df['g'])
mdf = md.fit()
print('completed in:', dt.now()-start)

completed in: 0:00:54.905572


In [ ]:
# mdf.summary()
# mdf.bse

In [17]:
reporting = pd.DataFrame()
reporting['coefs'] = mdf.params
reporting['stat'] = mdf.tvalues
reporting['p'] = mdf.pvalues
reporting['CI[.025, .975]'] = ['[{}]'.format(', '.join([np.format_float_scientific(x, precision=2) for x in ci.tolist()])) for ci in mdf.conf_int().values]

reporting['coefs'] = reporting['coefs'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['stat'] = reporting['stat'].apply(lambda x: np.format_float_scientific(x, precision=2))
reporting['p'] = reporting['p'].apply(lambda x: np.format_float_scientific(x, precision=2))

reporting.head(100)

,coefs,stat,p,"CI[.025, .975]"
Intercept,-3.7e-01,nan,nan,"[nan, nan]"
null[T.True],8.82e-02,1.75e+02,0.e+00,"[8.72e-02, 8.92e-02]"
nx,2.8e-01,1.74e+04,0.e+00,"[2.8e-01, 2.8e-01]"
ny,-2.47e-02,-1.53e+03,0.e+00,"[-2.47e-02, -2.47e-02]"
1 | groups_,-8.49e-09,-6.47e+00,1.00e-10,"[-1.11e-08, -5.92e-09]"
2 | lang_,3.72e-02,6.e+02,0.e+00,"[3.71e-02, 3.74e-02]"
Group Var,1.e+00,nan,nan,"[nan, nan]"


In [18]:
report_name = 'how-much-does-null-differ.csv'
report_name = os.path.join(DATA_LOC,report_name)

In [19]:
reporting.to_csv(
    report_name,
    encoding='utf-8'
)

In [20]:
reporting['Var'] = reporting.index.values

with open(report_name.replace('.csv', '2.txt'), 'w') as f:
    txt =  reporting[['Var', 'coefs', 'stat', 'p']].to_latex(index=False).replace('\\toprule', '\\hline').replace('\\midrule', '\\hline\\hline').replace('\\bottomrule', '\\hline')

    txt = txt.replace('\\\\', '\\\\\hline').replace('|', '$|$').replace('_delta', ' $\Delta$')
    f.write(txt)
    f.close()